# PHASE 0: Environment Setup & Safe Extraction

### PHASE 0.A: ENVIRONMENT SETUP

In [1]:
import sys
import os

# 1. Add the project root to the Python path to import 'src'
project_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if project_root not in sys.path:
    sys.path.append(project_root)

# 2. Import project configuration (Absolute paths)
from src import config

# 3. Import Data Science libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 4. Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("✅ Setup complete. Ready to load data.")

✅ Setup complete. Ready to load data.


### PHASE 0.B: DATA EXTRACTION

In [2]:

print("Loading MITMA mobility data... This might take a few seconds.\n")

try:
    # Load the compressed dataset
    df_mobility = pd.read_csv(
        config.PATH_MOBILITY, 
        sep='|', 
        compression='gzip', 
        encoding='utf-8',
        low_memory=False 
    )
    
    # Print the size of the dataset
    total_rows = df_mobility.shape[0]
    total_cols = df_mobility.shape[1]
    
    print(f"✅ Data loaded successfully!")
    print(f"📊 Dataset size: {total_rows:,} rows and {total_cols} columns.")
    
    # Show the first 5 rows to verify it loaded correctly
    display(df_mobility.head())
    
except Exception as e:
    print(f"❌ Error loading data: {e}")

Loading MITMA mobility data... This might take a few seconds.

✅ Data loaded successfully!
📊 Dataset size: 13,643,807 rows and 15 columns.


,fecha,periodo,origen,destino,distancia,actividad_origen,actividad_destino,estudio_origen_posible,estudio_destino_posible,residencia,renta,edad,sexo,viajes,viajes_km
0,20251015,6,01009_AM,01001,10-50,frecuente,casa,no,no,1,>15,NaN,NaN,7.000,111.909
1,20251015,16,01017_AM,01001,10-50,frecuente,casa,no,no,1,>15,NaN,NaN,6.272,80.938
2,20251015,14,01051,01001,10-50,frecuente,casa,no,no,1,>15,NaN,NaN,7.000,73.106
3,20251015,5,01058_AM,01001,10-50,frecuente,casa,no,no,1,>15,NaN,NaN,4.300,52.415
4,20251015,14,01058_AM,01001,10-50,frecuente,casa,no,no,1,>15,NaN,NaN,4.300,69.532


# PHASE 1: Data Audit, Optimization & Geographic Filtering

### PHASE 1: DATA AUDIT & MEMORY OPTIMIZATION

In [3]:
print("🔍 1. Initial Memory Audit (Before Optimization):")

df_mobility.info(verbose=False, memory_usage='deep')
print("=" * 50)

# Downcasting numeric columns to save memory
float_cols = df_mobility.select_dtypes(include=['float64']).columns
df_mobility[float_cols] = df_mobility[float_cols].astype('float32')

int_cols = df_mobility.select_dtypes(include=['int64']).columns
df_mobility[int_cols] = df_mobility[int_cols].astype('int32')

object_cols = df_mobility.select_dtypes(include=['object','str']).columns
df_mobility[object_cols] = df_mobility[object_cols].astype('string')

print("\n🔍 2. Final Memory Audit (After Optimization):")
df_mobility.info(verbose=False, memory_usage='deep')
print("=" * 50)
print("✅ Downcasting complete.")

🔍 1. Initial Memory Audit (Before Optimization):
<class 'pandas.DataFrame'>
RangeIndex: 13643807 entries, 0 to 13643806
Columns: 15 entries, fecha to viajes_km
dtypes: float64(2), int64(3), str(10)
memory usage: 2.2 GB

🔍 2. Final Memory Audit (After Optimization):
<class 'pandas.DataFrame'>
RangeIndex: 13643807 entries, 0 to 13643806
Columns: 15 entries, fecha to viajes_km
dtypes: float32(2), int32(3), string(10)
memory usage: 1.9 GB
✅ Downcasting complete.


### PHASE 1.B: COLUMN INSPECTION & GEOGRAPHIC FILTERING PREP

In [4]:

print("🔍 1. Inspecting Dataset Columns:")
columns_list = df_mobility.columns.tolist()
print(columns_list)
print("=" * 50)

print("\n🔍 2. Previewing the last 5 rows to understand the dataset:")
display(df_mobility.tail())

🔍 1. Inspecting Dataset Columns:
['fecha', 'periodo', 'origen', 'destino', 'distancia', 'actividad_origen', 'actividad_destino', 'estudio_origen_posible', 'estudio_destino_posible', 'residencia', 'renta', 'edad', 'sexo', 'viajes', 'viajes_km']

🔍 2. Previewing the last 5 rows to understand the dataset:


,fecha,periodo,origen,destino,distancia,actividad_origen,actividad_destino,estudio_origen_posible,estudio_destino_posible,residencia,renta,edad,sexo,viajes,viajes_km
13643802,20251015,13,5200108,5200108,2-10,no_frecuente,trabajo_estudio,no,no,52,>15,25-45,hombre,7.883,19.715000
13643803,20251015,20,5200106,5200108,2-10,no_frecuente,trabajo_estudio,no,no,52,10-15,45-65,<NA>,5.169,10.786000
13643804,20251015,18,5200108,5200108,2-10,no_frecuente,trabajo_estudio,no,no,52,>15,45-65,mujer,6.274,14.315000
13643805,20251015,23,0401301,5200108,>50,no_frecuente,trabajo_estudio,no,no,52,<10,45-65,<NA>,4.923,876.546021
13643806,20251015,12,2004503,FRI15,0.5-2,no_frecuente,trabajo_estudio,no,no,20,>15,45-65,hombre,6.822,8.632000


### PHASE 1.C: GEOGRAPHIC FILTERING (ISOLATING BARCELONA)

In [5]:
# Note: Column 'destino' means:
# 08  (Provincia de Barcelona)
# 019 (Municipio de Barcelona)
# 01  (Distrito, 1 = Ciutat Vella)

print("Applying Geographic Filter: Destination = Barcelona (INE: 08019)...\n")

# 1. Ensure the 'destino' column is treated as a string to use string methods
df_mobility['destino'] = df_mobility['destino'].astype(str)

# 2. Filter rows where the destination starts with '08019'
df_bcn = df_mobility[df_mobility['destino'].str.startswith('08019', na=False)].copy()

# 3. Calculate how much data we saved
initial_rows = df_mobility.shape[0]
bcn_rows = df_bcn.shape[0]
retained_pct = (bcn_rows / initial_rows) * 100

print(f"📉 Initial dataset size: {initial_rows:,} rows") 
print(f"🎯 Barcelona dataset size: {bcn_rows:,} rows")
print(f"✂️ We kept only {retained_pct:.2f}% of the data")

# 4. Delete the massive original dataframe to free up RAM
import gc # Garbage Collector library
del df_mobility
gc.collect()

print("\n✅ Original massive dataset deleted from memory.")

Applying Geographic Filter: Destination = Barcelona (INE: 08019)...

📉 Initial dataset size: 13,643,807 rows
🎯 Barcelona dataset size: 404,492 rows
✂️ We kept only 2.96% of the data

✅ Original massive dataset deleted from memory.


# PHASE 2: Missing Values (NULLS) & Business Logic

In [6]:

print("🔍 1. Auditing Missing Values (Nulls) in Barcelona data:")
# Calculate total nulls per column
null_counts = df_bcn.isnull().sum()
# Display only columns that actually have missing values
missing_data = null_counts[null_counts > 0]

if missing_data.empty:
    print("✅ No missing values in this dataset.")
else:
    print("⚠️ Missing values detected:")
    print(missing_data)
print("=" * 50)

# It's good practice to fill potential nulls with 0 to avoid errors.
df_bcn['viajes'] = df_bcn['viajes'].fillna(0)

print("\n🧮 2. Calculating Daily Foot Traffic per Zone...")

# Group by 'destino' and sum the 'viajes'
df_demand = df_bcn.groupby('destino')['viajes'].sum().reset_index()

# Rename columns to English for our database
df_demand.rename(columns={
    'destino': 'district_id', 
    'viajes': 'daily_foot_traffic'
}, inplace=True)

# Converting trips to pure integers (excluding decimals and scientific notation).
df_demand['daily_foot_traffic'] = df_demand['daily_foot_traffic'].astype(int)

# Sort the values from highest to lowest traffic
df_demand = df_demand.sort_values(by='daily_foot_traffic', ascending=False)

print("✅ Demand calculation complete! Here are the Top 5 most visited areas:")
display(df_demand.head())

🔍 1. Auditing Missing Values (Nulls) in Barcelona data:
⚠️ Missing values detected:
edad    31651
sexo    74312
dtype: int64

🧮 2. Calculating Daily Foot Traffic per Zone...
✅ Demand calculation complete! Here are the Top 5 most visited areas:


,district_id,daily_foot_traffic
1,0801902,1121181
9,0801910,796929
2,0801903,706707
4,0801905,603606
8,0801909,531179


# PHASE 3: EXPORTING THE DELIVERABLE (PRODUCTION READY)

In [7]:
print("💾 Saving the cleaned demand dataset for the final model...")

# 1. Directory defined in config.py
processed_dir = config.PROCESSED_DATA_DIR

# 2. Ensure the directory exists (it creates it if you haven't yet)
os.makedirs(processed_dir, exist_ok=True)

# 3. Define the file name
output_path = os.path.join(processed_dir, "01_mitma_demand_cleaned.csv")

# 4. Save the DataFrame to CSV without the pandas row index
df_demand.to_csv(output_path, index=False)

print(f"✅ Success! File securely saved at:")
print(f"📂 {output_path}")
print("\n🎉 EDA MOBILITY COMPLETE.")

💾 Saving the cleaned demand dataset for the final model...
✅ Success! File securely saved at:
📂 /home/marvin/repos/pj-geo-yield-ai/data/processed/01_mitma_demand_cleaned.csv

🎉 EDA MOBILITY COMPLETE.
